# Módulo 3 — Introducción a las series temporales

**Curso: Análisis y Pronóstico de Datos Mineros con Python**

> Reconocer componentes, rezagos, diferencias y estadísticas móviles; descomponer una serie.

---

### Cómo usar este notebook
1. Ábrelo en **Google Colab** y ejecuta las celdas **de arriba hacia abajo**.
2. Los datos se descargan solos desde el repositorio del curso; no tienes que subir nada.
3. Lee las salidas: cada bloque responde una pregunta concreta, no ejecutes por ejecutar.

In [ ]:
# Librerías base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.width', 120)

# --- Datos del curso -------------------------------------------------
# Los CSV viven en el repositorio del curso y se descargan solos.
# Si no hubiera internet, la función pide subir el archivo a mano.
REPO_DATOS = 'https://raw.githubusercontent.com/HishanFarfan/curso-datos-mineros/main/datos'

def cargar_datos(nombre, **kw):
    try:
        return pd.read_csv(f'{REPO_DATOS}/{nombre}', **kw)
    except Exception as e:
        print('No se pudo descargar desde GitHub:', e)
    try:
        from google.colab import files          # Colab: subir a mano
        print(f"Sube '{nombre}':")
        return pd.read_csv(next(iter(files.upload())), **kw)
    except ModuleNotFoundError:
        return pd.read_csv(nombre, **kw)         # local: archivo en el cwd

## 1. Carga de la serie preparada

Partimos de la versión ya limpia y ordenada (`_LIMPIO.csv`). En un flujo real usarías la salida del Módulo 1.

In [ ]:
df = cargar_datos('datos_proceso_planta_LIMPIO.csv', parse_dates=['Fecha'])
df = df.sort_values('Fecha').set_index('Fecha')
df = df.asfreq('h')   # eje horario regular; expone huecos como NaN
df.head()

In [ ]:
serie = df['Tonelaje_tph']
serie.plot(title='Serie horaria de tonelaje'); plt.ylabel('t/h'); plt.show()

## 2. Frecuencia y agregación

In [ ]:
serie.resample('D').mean().plot(title='Promedio diario de tonelaje')
plt.ylabel('t/h'); plt.show()

## 3. Rezagos (lags)

In [ ]:
tmp = pd.DataFrame({'X_t': serie,
                    'X_t-1': serie.shift(1),
                    'X_t-24': serie.shift(24)})
tmp.head(30)

In [ ]:
plt.scatter(serie.shift(1), serie, s=4, alpha=0.2)
plt.xlabel('X(t-1)'); plt.ylabel('X(t)'); plt.title('Dispersión con rezago 1'); plt.show()

## 4. Diferencias

In [ ]:
serie.diff().plot(title='Primera diferencia  ΔX_t = X_t - X_{t-1}'); plt.show()

In [ ]:
serie.diff(24).plot(title='Diferencia estacional (24 h)'); plt.show()

## 5. Estadísticas móviles

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
serie.plot(ax=ax, alpha=0.4, label='serie')
serie.rolling(24).mean().plot(ax=ax, label='media móvil 24 h')
serie.rolling(168).mean().plot(ax=ax, label='media móvil 168 h')
ax.legend(); plt.show()

In [ ]:
serie.rolling(24).std().plot(title='Desviación estándar móvil (24 h)'); plt.show()

## 6. Suavizado exponencial

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
serie.iloc[:500].plot(ax=ax, alpha=0.4, label='original')
serie.iloc[:500].ewm(alpha=0.1).mean().plot(ax=ax, label='EWM alpha=0.1')
serie.iloc[:500].ewm(alpha=0.4).mean().plot(ax=ax, label='EWM alpha=0.4')
ax.legend(); plt.show()

## 7. Descomposición

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
res = seasonal_decompose(serie.dropna(), model='additive', period=24)
res.plot(); plt.tight_layout(); plt.show()

In [ ]:
from statsmodels.tsa.seasonal import STL
stl = STL(serie.dropna(), period=24, robust=True).fit()
stl.plot(); plt.tight_layout(); plt.show()

El residuo de la descomposición: ¿es ruido o todavía tiene estructura? Eso se responde en el Módulo 4.

## Actividades sugeridas

1. Repite todo con `Recuperacion_pct` y compara la fuerza de la estacionalidad diaria.
2. Prueba `period=168` (semana) en la descomposición de la serie diaria.
3. ¿Qué tamaño de ventana móvil aísla mejor la deriva de fondo de la recuperación?
4. Grafica la media y la desviación móviles juntas: ¿el proceso es igual de estable todo el periodo?

---
## Cierre

Ya vemos tendencia, estacionalidad, rezagos y cambios locales. Falta cuantificar la dependencia (ACF/PACF) y decidir si la serie es estacionaria: Módulo 4.